# Borrower Reliability Analysis

## Open the table and explore the general information about the data

**Task 1. Import the pandas library. Read the data from the CSV file into a DataFrame and save it to the variable `data`. File path:**

`/datasets/data.csv`

In [1]:
import pandas as pd

try:
    data = pd.read_csv('/datasets/data.csv')
except:
    data = pd.read_csv('https://code.s3.yandex.net/datasets/data.csv')

**Task 2. Display the first 20 rows of the `data` DataFrame.**

In [2]:
data.head(20)

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,-8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453,покупка жилья
1,1,-4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102,приобретение автомобиля
2,0,-5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297,покупка жилья
3,3,-4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628.550329,дополнительное образование
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616.077870,сыграть свадьбу
5,0,-926.185831,27,высшее,0,гражданский брак,1,M,компаньон,0,255763.565419,покупка жилья
6,0,-2879.202052,43,высшее,0,женат / замужем,0,F,компаньон,0,240525.971920,операции с жильем
7,0,-152.779569,50,СРЕДНЕЕ,1,женат / замужем,0,M,сотрудник,0,135823.934197,образование
8,2,-6929.865299,35,ВЫСШЕЕ,0,гражданский брак,1,F,сотрудник,0,95856.832424,на проведение свадьбы
9,0,-2188.756445,41,среднее,1,женат / замужем,0,M,сотрудник,0,144425.938277,покупка жилья для семьи


**Task 3. Display the basic information about the DataFrame using the `info()` method.**

In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   days_employed     19351 non-null  float64
 2   dob_years         21525 non-null  int64  
 3   education         21525 non-null  object 
 4   education_id      21525 non-null  int64  
 5   family_status     21525 non-null  object 
 6   family_status_id  21525 non-null  int64  
 7   gender            21525 non-null  object 
 8   income_type       21525 non-null  object 
 9   debt              21525 non-null  int64  
 10  total_income      19351 non-null  float64
 11  purpose           21525 non-null  object 
dtypes: float64(2), int64(5), object(5)
memory usage: 2.0+ MB


## Data preprocessing

### Removing Missing Values

**Task 4. Display the number of missing values for each column. Use a combination of two methods.**

In [4]:
data.isna().sum()

children               0
days_employed       2174
dob_years              0
education              0
education_id           0
family_status          0
family_status_id       0
gender                 0
income_type            0
debt                   0
total_income        2174
purpose                0
dtype: int64

**Task 5. There are missing values in two columns. One of them is `days_employed`. You will handle the missing values in this column in the next step. The other column with missing values is `total_income`, which contains income data. Since income is most strongly influenced by the type of employment, the missing values in this column should be filled with the median income for each employment type from the `income_type` column. For example, if a person has the employment type `employee`, the missing value in their `total_income` field should be filled with the median income among all records with the same type.**

In [5]:
for t in data['income_type'].unique():
    data.loc[(data['income_type'] == t) & (data['total_income'].isna()), 'total_income'] = \
    data.loc[(data['income_type'] == t), 'total_income'].median()

**Task 4.1. Translate some of the dataset's columns.**

In [7]:
# Create a dictionary to translate the income_type column
income_type_dict = {
    'безработный': 'unemployed',
    'в декрете': 'on maternity leave',
    'госслужащий': 'state servant',
    'компанъон': 'partner',  # Verify if "компаньон" should be spelled differently
    'пенсионер': 'retiree',
    'предприниматель': 'entrepreneur',
    'сотрудник': 'employee',
    'студент': 'student'
}

# Create a dictionary to translate the family_status column
family_status_dict = {
    'Не женат / не замужем': 'unmarried / single',
    'в разводе': 'divorced',
    'вдовец / вдова': 'widowed',
    'гражданский брак': 'civil marriage',
    'женат / замужем': 'married'
}

# Overwrite the existing columns
data['income_type'] = data['income_type'].map(income_type_dict)
data['family_status'] = data['family_status'].map(family_status_dict)

# Display the first 5 rows to verify the result
data.head()


,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,8437.673028,42,высшее,0,married,0,F,employee,0,253875.639453,покупка жилья
1,1,4024.803754,36,среднее,1,married,0,F,employee,0,112080.014102,приобретение автомобиля
2,0,5623.422610,33,Среднее,1,married,0,M,employee,0,145885.952297,покупка жилья
3,3,4124.747207,32,среднее,1,married,0,M,employee,0,267628.550329,дополнительное образование
4,0,340266.072047,53,среднее,1,civil marriage,1,F,retiree,0,158616.077870,сыграть свадьбу


### Handling abnormal values

**Task 6. The data may contain artifacts (anomalies) — values that do not reflect reality and appeared due to some error. An example of such an artifact is the negative number of days of work experience in the `days_employed` column. This is common in real-world datasets. Handle the values in this column by converting all negative values to positive using the `abs()` method.**

In [6]:
data['days_employed'] = data['days_employed'].abs()

**Task 7. For each type of employment, display the median value of work experience `days_employed` in days.**

In [8]:
data.groupby('income_type')['days_employed'].agg('median')

income_type
employee                1574.202821
entrepreneur             520.848083
on maternity leave      3296.759962
retiree               365213.306266
state servant           2689.368353
student                  578.751554
unemployed            366413.652744
Name: days_employed, dtype: float64

Two types (unemployed and retirees) will result in abnormally high values. These values are difficult to correct, so leave them as they are.

**Task 8. Display the list of unique values in the `children` column.**

In [9]:
data['children'].unique()

array([ 1,  0,  3,  2, -1,  4, 20,  5], dtype=int64)

**Task 9. There are two anomalous values in the `children` column. Remove the rows containing these anomalous values from the `data` DataFrame.**

In [10]:
data = data[(data['children'] != -1) & (data['children'] != 20)]

**Task 10. Display the list of unique values in the `children` column once again to ensure the anomalies have been removed.**

In [11]:
data['children'].unique()

array([1, 0, 3, 2, 4, 5], dtype=int64)

### Handling Missing Values (continued)

**Task 11. Fill in the missing values in the `days_employed` column with the median values for each employment type (`income_type`).**

In [12]:
for t in data['income_type'].unique():
    data.loc[(data['income_type'] == t) & (data['days_employed'].isna()), 'days_employed'] = \
    data.loc[(data['income_type'] == t), 'days_employed'].median()

**Task 12. Make sure all missing values have been filled. Double-check yourself and display the number of missing values for each column once again using two methods.**

In [13]:
data.isna().sum()

children               0
days_employed        504
dob_years              0
education              0
education_id           0
family_status          0
family_status_id       0
gender                 0
income_type         5054
debt                   0
total_income           0
purpose                0
dtype: int64

### Changing Data Types

**Task 13. Convert the float data type in the `total_income` column to an integer using the `astype()` method.**

In [14]:
data['total_income'] = data['total_income'].astype(int)

### Handling Duplicates

**Task 14. Handle the implicit duplicates in the `education` column. This column contains identical values written differently using uppercase and lowercase letters. Convert all values to lowercase.**

In [15]:
data['education'] = data['education'].str.lower()

**Task 15. Display the number of duplicate rows in the data. If such rows are present, remove them.**

In [16]:
data.duplicated().sum()

71

In [17]:
data = data.drop_duplicates()

### Data Categorization

**Task 16. Based on the ranges listed below, create a new column in the `data` DataFrame called `total_income_category` with the following categories:**

- 0–30,000 — `'E'`  
- 30,001–50,000 — `'D'`  
- 50,001–200,000 — `'C'`  
- 200,001–1,000,000 — `'B'`  
- 1,000,001 and above — `'A'`  

**For example, a borrower with an income of 25,000 should be assigned category `'E'`, and a client earning 235,000 — category `'B'`.  
Use a custom function named `categorize_income()` and the `apply()` method.**

In [18]:
def categorize_income(income):
    try:
        if 0 <= income <= 30000:
            return 'E'
        elif 30001 <= income <= 50000:
            return 'D'
        elif 50001 <= income <= 200000:
            return 'C'
        elif 200001 <= income <= 1000000:
            return 'B'
        elif income >= 1000001:
            return 'A'
    except:
        pass

In [19]:
data['total_income_category'] = data['total_income'].apply(categorize_income)

**Task 17. Display the list of unique loan purposes from the `purpose` column.**

In [20]:
data['purpose'].unique()

array(['покупка жилья', 'приобретение автомобиля',
       'дополнительное образование', 'сыграть свадьбу',
       'операции с жильем', 'образование', 'на проведение свадьбы',
       'покупка жилья для семьи', 'покупка недвижимости',
       'покупка коммерческой недвижимости', 'покупка жилой недвижимости',
       'строительство собственной недвижимости', 'недвижимость',
       'строительство недвижимости', 'на покупку подержанного автомобиля',
       'на покупку своего автомобиля',
       'операции с коммерческой недвижимостью',
       'строительство жилой недвижимости', 'жилье',
       'операции со своей недвижимостью', 'автомобили',
       'заняться образованием', 'сделка с подержанным автомобилем',
       'получение образования', 'автомобиль', 'свадьба',
       'получение дополнительного образования', 'покупка своего жилья',
       'операции с недвижимостью', 'получение высшего образования',
       'свой автомобиль', 'сделка с автомобилем',
       'профильное образование', 'высшее об

**Task 18. Create a function that, based on the values in the `purpose` column, generates a new column called `purpose_category` with the following categories:**

- `'car operations'`  
- `'real estate operations'`  
- `'wedding expenses'`  
- `'education'`  

**For example, if the `purpose` column contains the substring `'на покупку автомобиля'`, the `purpose_category` column should contain `'car operations'`.**  

**Use a custom function named `categorize_purpose()` and the `apply()` method. Explore the values in the `purpose` column to determine which substrings will help you accurately classify each entry.**

In [21]:
def categorize_purpose(row):
    try:
        if 'автом' in row:
            return 'car operations'
        elif 'жил' in row or 'real estate' in row:
            return 'real estate operations'
        elif 'свад' in row:
            return 'wedding expenses'
        elif 'образов' in row:
            return 'education'
    except:
        return 'no category'


In [22]:
data['purpose_category'] = data['purpose'].apply(categorize_purpose)

### Step 3. Explore the data and answer the questions

#### 3.1 Is there a relationship between the number of children and on-time loan repayment?

In [23]:
data_grouped_children = data.groupby('children').agg({'debt': ['count', 'sum']})
data_grouped_children['debt_percent'] = (data_grouped_children['debt']['sum'] / data_grouped_children['debt']['count']) * 100

data_grouped_children

debt       debt_percent
          count   sum             
children                          
0         14091  1063     7.543822
1          4808   444     9.234609
2          2052   194     9.454191
3           330    27     8.181818
4            41     4     9.756098
5             9     0     0.000000

**Conclusion:**

Borrowers without children have the lowest default rate (7.5%), indicating a higher likelihood of repaying the loan on time compared to borrowers with children.

As the number of children increases, the default rate generally rises (9.2% for those with 1 child, 9.6% for 2 children, and 9.8% for 4 children). However, for borrowers with 3 children, the default rate drops to 8.18%, contradicting the assumption that more children always lead to a higher credit risk.

For borrowers with 5 children, the default rate is 0. This is likely due to the very small sample size (9 observations).

It is important to note that the sample sizes differ across groups based on the number of children. The largest sample belongs to borrowers without children (14,091), which makes conclusions about childless borrowers more statistically reliable. The reliability of conclusions for groups with more children (especially those with 4 or 5 children) may be lower due to smaller sample sizes, which increase the chance of random fluctuations.

#### 3.2 Is there a relationship between marital status and on-time loan repayment?

In [24]:
data_grouped_fam_st = data.groupby('family_status').agg({'debt': ['count', 'sum']})
data_grouped_fam_st['debt_percent'] = (data_grouped_fam_st['debt']['sum'] / data_grouped_fam_st['debt']['count']) * 100

data_grouped_fam_st

debt      debt_percent
                    count  sum             
family_status                              
civil marriage       4134  385     9.313014
divorced             1189   84     7.064760
married             12261  927     7.560558
unmarried / single   2796  273     9.763948
widowed               951   63     6.624606

**Conclusion:**  
Based on the provided data on the share of overdue loans across different marital status categories, it can be concluded that there is a certain relationship between marital status and a borrower's ability to repay a loan on time. Borrowers who are not in a registered marriage show a higher percentage of defaults compared to those who are officially married, divorced, or widowed.

Among borrowers not in a registered marriage ("single" and "civil marriage"), there is a higher share of loan defaults (9.7% and 9.3%, respectively).

Borrowers who are officially married, as well as those who are divorced, show lower default rates (7.6% and 7.1%, respectively).

The lowest default rate is observed among widows and widowers (6.6%).

It is also important to note the different sample sizes for each group. For groups with a larger number of observations ("married", "civil marriage"), the results are more reliable. Groups with fewer observations, such as "widowed" (951) and "divorced" (1,189), have smaller sample sizes (though still sufficient for conclusions). This may also indicate an uneven distribution of marital statuses in the overall population or among borrowers applying for loans.

#### 3.3 Is there a relationship between income level and on-time loan repayment?

In [25]:
data_grouped_income = data.groupby('total_income_category').agg({'debt': ['count', 'sum']})
data_grouped_income['debt_percent'] = (data_grouped_income['debt']['sum'] / data_grouped_income['debt']['count']) * 100

data_grouped_income

debt       debt_percent
                       count   sum             
total_income_category                          
A                         25     2     8.000000
B                       5014   354     7.060231
C                      15921  1353     8.498210
D                        349    21     6.017192
E                         22     2     9.090909

**Conclusion:**  
Borrowers in category E (the lowest income level) have the highest loan default rate (9.1%).  
Borrowers in the middle-income categories (A, B, C) show varying default rates: 8% for category A, 7.1% for B, and 8.5% for C.  
The lowest default rate is observed in category D (6%).

Overall, it is difficult to draw definitive conclusions about the relationship between income level and loan repayment due to the small sample sizes in categories A and E (22 and 25 cases, respectively). The number of observations in category D is also relatively small (349).  
However, for categories B and C, which have a significant number of observations, a trend can be seen: borrowers with higher income (B) show a lower default rate compared to category C.

#### 3.4 How do different loan purposes affect on-time repayment?

In [26]:
data_grouped_purpose = data.groupby('purpose_category').agg({'debt': ['count', 'sum']})
data_grouped_purpose['debt_percent'] = (data_grouped_purpose['debt']['sum'] / data_grouped_purpose['debt']['count']) * 100

data_grouped_purpose

debt      debt_percent
                       count  sum             
purpose_category                              
car operations          4279  400     9.347978
education               3988  369     9.252758
real estate operations  5659  397     7.015374
wedding expenses        2313  183     7.911803

**Conclusion:**  
The purpose of the loan has a noticeable impact on on-time repayment. Loans for real estate operations show the lowest default rate (7.3%).  
At the same time, loans for education and car-related purposes demonstrate higher default risks (9.3% for both).  
Wedding-related loans fall in between, with a default rate of 8%.

#### 3.5 Provide possible reasons for the appearance of missing values in the original data.

*Answer:* The original data contains missing values in the columns `'total_income'` (income data) — 2,174 entries, and `'days_employed'` (employment duration in days) — 2,174 entries.

In this case, the missing values in income and employment data may be due to the absence of employment among some borrowers (such as those in the "unemployed" and "retired" categories), meaning they may not have been able to provide information about their income or work experience.

Other possible reasons for missing data include technical issues, such as errors during data entry, copying, reading, or changes in file format.

#### 3.6 Explain why filling in missing values with the median is the best solution for quantitative variables.

*Answer:* The median is less sensitive to outliers and anomalous values compared to the mean. In cases where the data distribution is highly skewed or contains outliers, using the median helps prevent distortion in the imputed values.

### Step 4: Overall conclusion.

#### Data Preprocessing with a Description of Identified Issues and Solutions

During data preprocessing, the following key issues were identified:

**Missing Values:**  
Missing values were found in the columns `'total_income'` and `'days_employed'`. This may be due to a lack of employment among certain borrower categories or technical errors during data collection. To address this issue, it was decided to fill in the missing values with the median values for the corresponding categories. This approach preserves the distribution structure of the data and ensures the reliability of subsequent analysis.

**Artifacts (Anomalies):**  
Values were found that do not reflect reality and appeared due to some error. For example, negative values for days of work experience in the `'days_employed'` column. All negative values were replaced with their absolute values.  
In the `'children'` column, two anomalous values were encountered. The rows containing these values were removed.

**Duplicates:**  
Duplicates were detected, which may have been caused by data entry errors. These duplicate rows were removed.

**Small Sample Sizes in Some Groups:**  
When analyzing the relationship between on-time loan repayment and factors such as income level and the number of children, issues with small sample sizes were noted in some groups. For instance, there is very little data on borrowers with the highest and lowest incomes (22 and 25 cases, respectively) as well as on families with 5 children (9 observations). This makes it challenging to draw precise conclusions for these categories.

---

#### Answers to the Project Objectives

Based on the analysis conducted, the following conclusions were drawn:

**Impact of the Number of Children on On-Time Loan Repayment:**  
Borrowers without children have the lowest default rate (7.5%), indicating a higher likelihood of repaying loans on time compared to borrowers with children. As the number of children increases, the default rate generally rises (9.2% for those with 1 child, 9.6% for those with 2 children, and 9.8% for those with 4 children), although there are exceptions (for example, borrowers with 3 children have a default rate of 8.18%), highlighting that this relationship is not strictly linear.

**Impact of Marital Status on On-Time Loan Repayment:**  
Borrowers who are not in an official marriage exhibit a higher default rate (for the categories "Not married" – 9.7% and "Civil marriage" – 9.3%) compared to those who are officially married, divorced, or widowed.  
Borrowers who are officially married ("married") and divorced have lower default rates (7.6% and 7.1%, respectively).  
The lowest default rate is observed among widows and widowers (6.6%).

**Impact of Income Level on On-Time Loan Repayment:**  
It is challenging to draw definitive conclusions about the effect of income level on loan repayment due to the limited sample sizes in some income categories. Overall, there is a trend toward a higher default rate among borrowers with the lowest income (9.1%), while borrowers with higher and medium incomes (8% for category A, 7.1% for category B, and 8.5% for category C) demonstrate better repayment behavior. The lowest default rate is observed in category D (6%).

**Impact of Loan Purpose on On-Time Loan Repayment:**  
Loan purposes have a noticeable impact on repayment behavior. Loans for real estate operations have the lowest default rate (7.3%), while loans for education and car-related purposes exhibit higher default rates (9.3% for both). Loans for wedding expenses fall in between, with a default rate of 8%.

---

#### Recommendations for the Client

**Improve Data Verification:**  
Implement a system that automatically checks whether all necessary data have been entered to avoid missing values and input errors. This could include automated form completeness checks and warnings to users if required fields are left blank. For borrowers without official employment or who have difficulty providing income and work experience data, consider adding a special category such as "No data" or "Not applicable."

**Revise Credit Policies:**  
Adapt credit products to suit different borrower groups. For instance, consider introducing more flexible credit terms for specific categories.

**Offer Financial Literacy Courses:**  
Organize money management courses, particularly for borrowers taking loans for education, car purchases, or those with children, to help them better manage their finances.